# Dependancies 

In [1]:
import os
from openpyxl import Workbook
import matplotlib.pyplot as plt
import pandas as pd

# Something

In [2]:
import ME_modules.MakeWriteExcel as MWE
import ME_modules.Folder_n_File_Utilities as fnfu
import  ME_modules.ME_austin_EIS.chip_processor as MCP
processed_chips_folder_path = None

# Main

In [3]:


def main(use_hardcoded_path=False, hardcoded_path=None):
    # 1. Setup Phase
    # Create the initial workbooks structure
    processed_chips_folder = "Processed Chips for PCAI1ia"
    MWE.create_chips_folder_and_workbooks()

    # Determine the master folder path
    if use_hardcoded_path and hardcoded_path:
        master_folder = hardcoded_path
        print(f"Using hardcoded path: {master_folder}")
    else:
        master_folder = fnfu.select_folder()
        if not master_folder:
            print("No folder selected. Exiting.")
            return

    # Set up output folder and logging
    output_folder = os.path.join(master_folder, "Processed_Chip_Data")
    os.makedirs(output_folder, exist_ok=True)
    log_file = os.path.join(output_folder, "analysis_log.txt")
    open(log_file, 'w').close()

    # 2. Chip Validation Phase
    chip_data = fnfu.find_valid_chips(master_folder)
    valid_chips, invalid_chips = fnfu.match_chip_files(chip_data)
    
    fnfu.log_and_print("A valid chip must contain: water.csv, buffer.csv, dna.csv, cyst.csv, cas.xlsx", log_file)
    fnfu.log_and_print(f"Found {len(chip_data)} chip groups in all subfolders.", log_file)
    
    if not valid_chips:
        fnfu.log_and_print("No valid chips found.", log_file)
        return
    
    if invalid_chips:
        fnfu.log_and_print("Invalid chips found. Missing files:", log_file)
        for chip, missing in sorted(invalid_chips.items(), key=lambda x: int(x[0])):
            missing_types = [key for key in ["water", "buffer", "dna", "cyst", "cas"] if key not in missing]
            fnfu.log_and_print(f"Chip {chip} is missing: {', '.join(missing_types)}", log_file)

    # 3. Chip Selection Phase
    fnfu.log_and_print(f"Valid chips found: {sorted(valid_chips.keys(), key=int)}", log_file)
    chip_choice = input("Enter chip number to process or 'all' to process all valid chips: ").strip()
    
    chips_to_process = []
    if chip_choice.lower() == 'all':
        chips_to_process = sorted(valid_chips.keys(), key=int)
    elif chip_choice in valid_chips:
        chips_to_process = [chip_choice]
    else:
        fnfu.log_and_print(f"Invalid selection: {chip_choice}", log_file)
        return

    # 4. Processing Phase
    for chip in chips_to_process:
        chip_paths = valid_chips[chip]
        fnfu.log_and_print(f"\nProcessing Chip {chip}...", log_file)
        
        try:
            # Process the Excel file for the chip
            chip_results = MCP.process_chip_excel_only(chip, chip_paths)

            if not chip_results:
                fnfu.log_and_print(f"No valid results for Chip {chip}", log_file)
                continue

            # Write results into correct workbook/sheets
            MCP.write_chip_results_to_workbook(chip_results, processed_chips_folder)
            fnfu.log_and_print(f"Successfully processed and wrote results for Chip {chip}", log_file)
            
        except Exception as e:
            fnfu.log_and_print(f"Error processing Chip {chip}: {str(e)}", log_file)
            continue
    
    print(f"MCP.all_raw_names_logged:{MCP.all_raw_names_logged}")
    fnfu.log_and_print("\nProcessing completed!", log_file)

    output_folder = os.path.join(master_folder, "Processed_Chip_Data")
    
    return os.path.abspath(processed_chips_folder)


In [4]:
if __name__ == "__main__":
    processed_chips_folder_path = main()

A valid chip must contain: water.csv, buffer.csv, dna.csv, cyst.csv, cas.xlsx
Found 26 chip groups in all subfolders.
Valid chips found: ['21', '22', '23', '26', '27', '32', '33', '35', '36', '37', '39', '40', '41', '43', '44', '45', '46', '47', '48', '52', '53', '54', '55', '56', '57', '59']

Processing Chip 21...

=== DEBUG: Starting analysis for cycle 1 ===
DataFrame shape: (381, 6)
Columns: ['Frequency(Hz)', 'AC Status', 'Rs', 'Cp', 'X', 'PH']
find_transition_point_optimized: n=381, x_range=(269.77,4287.28)
  curvature_idx=41
  derivative_idx=24
  min_idx=378
      Radius_of_curvature range: [4.677e-01, 1.522e+05]
  radius_threshold_idx=23, thresh=1.895e+03
  ransac_tail_idx=318 (tail_len=63, inlier_ratio>=0.7)
  candidates: [23, 24, 41, 318, 378]
    candidate 23 -> score 1.176158e+05
    candidate 24 -> score 1.172544e+05
    candidate 41 -> score 1.111678e+05
    candidate 318 -> score 4.818629e+04
    candidate 378 -> score 1.057408e+03
  chosen transition_idx=378 with score=1.

In [5]:
print("Done")

Done


# Create Comparitive Graphs

In [6]:
import os
import pandas as pd
import numpy as np
import tkinter as tk
from tkinter import filedialog
import matplotlib.pyplot as plt

'''

Min-max normalization (scale to 0-1):
This method rescales the data so that the minimum value becomes 0 and the maximum becomes 1. 
It's useful for comparing datasets with different ranges or units. The formula is: (value - min) / (max - min).

Z-score normalization (subtract mean, divide by std):
This method standardizes the data by subtracting the mean and dividing by the standard deviation. 
It transforms the data into units of standard deviations from the mean, centering the distribution around zero. 
This is helpful for identifying outliers or comparing data across different scales.

Normalize to starting point (e.g., divide all values by the first time point value):
This method divides all values by the first data point in the series, effectively setting the starting point to 1. 
It's often used when studying relative changes over time. 
This normalization highlights trends without focusing on absolute values.

'''

if processed_chips_folder_path == None:
    root = tk.Tk()
    root.withdraw()
    processed_chips_folder_path = filedialog.askdirectory(title="Select folder with Chip Excel files")

# Normalization options
normalize_min_max = True
normalize_z_score = True
normalize_start = True

CHIP_INFO = {
    #"CHIP TEMPLATE" : ["Cas_{complex} or Cas_{only}","{0.5} or {1} or {5} Conentration of MgCl2", "{HU} protein or {SCDU} protein"],
    'Chip 21': ["Cas_complex"   ,"0.5"   ,"HU"     ],
    'Chip 22': ["Cas_complex"   ,"0.5"   ,"HU"     ],
    'Chip 23': ["Cas_complex"   ,"5"     ,"SCDU"   ],
    'Chip 26': ["Cas_complex"   ,"5"     ,"HU"     ],
    'Chip 27': ["Cas_complex"   ,"5"     ,"HU"     ],
    'Chip 32': ["Cas_complex"   ,"5"     ,"HU"     ],
    'Chip 33': ["Cas_only"      ,"1"     ,"HU"     ],
    'Chip 35': ["Cas_complex"   ,"1"     ,"HU"     ],
    'Chip 36': ["Cas_only"      ,"1"     ,"HU"     ],
    'Chip 37': ["Cas_complex"   ,"1"     ,"HU"     ],
    'Chip 39': ["Cas_complex"   ,"1"     ,"HU"     ],
    'Chip 40': ["Cas_complex"   ,"1"     ,"HU"     ],
    'Chip 41': ["Cas_complex"   ,"1"     ,"SCDU"   ],
    'Chip 43': ["Cas_complex"   ,"5"     ,"SCDU"   ],
    'Chip 44': ["Cas_complex"   ,"5"     ,"SCDU"   ],
    'Chip 45': ["Cas_only"      ,"5"     ,"HU"     ],
    'Chip 46': ["Cas_complex"   ,"1"     ,"SCDU"   ],
    'Chip 47': ["Cas_only"      ,"1"     ,"HU"     ],
    'Chip 48': ["cas_complex"   ,"5"     ,"SCDU"   ],
    'Chip 52': ["Cas_complex"   ,"5"     ,"HU"     ],
    'Chip 53': ["Cas_complex"   ,"1"     ,"SCDU"   ],
    'Chip 54': ["Cas_only"      ,"5"     ,"HU"     ],
    'Chip 55': ["Cas_complex"   ,"1"     ,"SCDU"   ],
    'Chip 56': ["Cas_only"      ,"5"     ,"HU"     ],
    'Chip 57': ["Cas_only"      ,"5"     ,"HU"     ],
    'Chip 59': ["Cas_complex"   ,"0.5"   ,"HU"     ]
}

# Constants
workbook_names = [f"Chip {n}" for n in [21,22,23,26,27,32,33,35,36,37,39,40,41,43,44,45,46,47,48,52,53,54,55,56,57,59]]
worksheet_names = ["0pM_asso", "0pM_disso", "100pM_asso", "100pM_disso", "1nM_asso", "1nM_disso", "10nM_asso", "10nM_disso", "100nM_asso", "100nM_disso"]
headers = [ 'time(mins)', 'delta Rct-a', 'delta Rct-d', 'Cp1', 'Ph1', 'Slope 1', 'Slope 2', 'Slope 3', 'Slope 4', 'Slope 5', 'Angle', 'Cp_exp-a', 'Cp_exp-b', 'Ph_slope', 'Ph_peak', 'Area Cp', 'Area Ph', 'Area Slope', 'Area Rs-direct', 'Area Rs-Para', '','','linear_eq_m','linear_eq_b','Rs','delta Rct-i','Q','n' ]

# Use the returned path to find Excel files
folder_path = processed_chips_folder_path

if not folder_path:
    print("No folder selected. Exiting.")
    exit()

# Read and combine all data
all_data = []
for file_name in os.listdir(folder_path):
    file_path = os.path.join(folder_path, file_name)
    chip_name = os.path.splitext(file_name)[0]

    if chip_name not in workbook_names or not file_path.endswith(".xlsx"):
        continue

    xl = pd.ExcelFile(file_path)
    for sheet in xl.sheet_names:
        if sheet not in worksheet_names:
            continue

        df = xl.parse(sheet, usecols=lambda col: col in headers)
        df['chip'] = chip_name
        df['sheet'] = sheet
        all_data.append(df)

if not all_data:
    print("No valid data found.")
    exit()

df_all = pd.concat(all_data, ignore_index=True)

# Add CHIP_INFO metadata
chip_info_df = pd.DataFrame.from_dict(CHIP_INFO, orient='index', columns=['Type', 'Concentration', 'Protein']).reset_index().rename(columns={'index': 'chip'})
df_all = df_all.merge(chip_info_df, on='chip', how='left')

# Define plot groups
plots = {
    "Plot 1": [("Cas_complex", "5", "HU"), ("Cas_only", "5", "HU")],
    "Plot 2": [("Cas_complex", "5", "SCDU"), ("Cas_complex", "5", "HU")],
    "Plot 3": [("Cas_complex", "0.5", "HU"), ("Cas_complex", "1", "HU"), ("Cas_complex", "5", "HU")],
    "Plot 4": [("Cas_complex", "0.5", "SCDU"), ("Cas_complex", "1", "SCDU"), ("Cas_complex", "5", "SCDU")]
}

y_vars = ['delta Rct-a', 'linear_eq_m']

# Normalization methods
normalizations = {
    'min-max': lambda y, min_val, max_val: (y - min_val) / (max_val - min_val) if (max_val - min_val) != 0 else np.nan,
    'z-score': lambda y, mean, std: (y - mean) / std if std != 0 else np.nan,
    'start': lambda y, start_val: y / start_val if start_val != 0 else np.nan
}

# Create main output folder
main_output_folder = os.path.join(folder_path, "comparison graphs")
os.makedirs(main_output_folder, exist_ok=True)

# Create folders per normalization, per concentration, per worksheet
for norm_name, norm_func in normalizations.items():
    norm_folder = os.path.join(main_output_folder, norm_name)
    os.makedirs(norm_folder, exist_ok=True)

    for y_var in y_vars:
        for plot_name, groups in plots.items():
            for worksheet in worksheet_names:
                # Extract concentration prefix (e.g., '0pM' from '0pM_asso')
                concentration = worksheet.split("_")[0]
                condition = worksheet  # e.g., '0pM_asso'

                # Create subfolders: concentration folder + condition folder
                conc_folder = os.path.join(norm_folder, concentration)
                condition_folder = os.path.join(conc_folder, condition)
                os.makedirs(condition_folder, exist_ok=True)

                plt.figure(figsize=(8,6))

                for g in groups:
                    subset = df_all[
                        (df_all['Type'] == g[0]) &
                        (df_all['Concentration'] == g[1]) &
                        (df_all['Protein'] == g[2]) &
                        (df_all['sheet'] == worksheet)
                    ]

                    if subset.empty:
                        print(f"No data for {g} in {plot_name} ({worksheet})")
                        continue

                    grouped = subset.groupby('time(mins)')[y_var].mean().reset_index()
                    if grouped.empty:
                        print(f"No data for {g} in {plot_name} ({worksheet})")
                        continue

                    min_val = grouped[y_var].min()
                    max_val = grouped[y_var].max()
                    mean_val = grouped[y_var].mean()
                    std_val = grouped[y_var].std()
                    first_val = grouped[y_var].iloc[0] if not grouped.empty else 0

                    if norm_name == 'min-max':
                        norm_y = norm_func(grouped[y_var], min_val, max_val)
                    elif norm_name == 'z-score':
                        norm_y = norm_func(grouped[y_var], mean_val, std_val)
                    elif norm_name == 'start':
                        norm_y = norm_func(grouped[y_var], first_val)
                    else:
                        norm_y = grouped[y_var]

                    plt.plot(grouped['time(mins)'], norm_y, label=f"{g}")

                plt.xlabel('time(mins)')
                plt.ylabel(f"{y_var} ({norm_name})")
                plt.title(f"{plot_name} - {y_var} ({worksheet}, {norm_name})")
                plt.legend()
                plt.tight_layout()

                # Save figure in the appropriate subfolder
                safe_plot_name = plot_name.replace(" ", "_")
                safe_y_var = y_var.replace(" ", "_").replace("-", "_")
                safe_worksheet = worksheet.replace(" ", "_").replace("-", "_")
                filename = f"{safe_plot_name}_{safe_y_var}_{safe_worksheet}.png"
                plt.savefig(os.path.join(condition_folder, filename))
                plt.close()


C:\Users\austi\AppData\Local\Temp\ipykernel_24104\562289110.py:101: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat(all_data, ignore_index=True)


No data for ('Cas_complex', '0.5', 'SCDU') in Plot 4 (0pM_asso)
No data for ('Cas_complex', '0.5', 'SCDU') in Plot 4 (0pM_disso)
No data for ('Cas_complex', '0.5', 'SCDU') in Plot 4 (100pM_asso)
No data for ('Cas_complex', '0.5', 'SCDU') in Plot 4 (100pM_disso)
No data for ('Cas_complex', '0.5', 'SCDU') in Plot 4 (1nM_asso)
No data for ('Cas_complex', '0.5', 'SCDU') in Plot 4 (1nM_disso)
No data for ('Cas_complex', '0.5', 'SCDU') in Plot 4 (10nM_asso)
No data for ('Cas_complex', '0.5', 'SCDU') in Plot 4 (10nM_disso)
No data for ('Cas_complex', '0.5', 'SCDU') in Plot 4 (100nM_asso)
No data for ('Cas_complex', '0.5', 'SCDU') in Plot 4 (100nM_disso)
No data for ('Cas_complex', '0.5', 'SCDU') in Plot 4 (0pM_asso)
No data for ('Cas_complex', '0.5', 'SCDU') in Plot 4 (0pM_disso)
No data for ('Cas_complex', '0.5', 'SCDU') in Plot 4 (100pM_asso)
No data for ('Cas_complex', '0.5', 'SCDU') in Plot 4 (100pM_disso)
No data for ('Cas_complex', '0.5', 'SCDU') in Plot 4 (1nM_asso)
No data for ('Cas_c

In [7]:
print("Done")

Done


# Big Excel for Deepta

In [8]:
import os
import pandas as pd
import numpy as np
import tkinter as tk
from tkinter import filedialog
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

# --- Define path variable first ---
# processed_chips_folder_path = None
plot_boolean = False  # set True if you want plots

# GUI to select folder
if processed_chips_folder_path is None:
    root = tk.Tk()
    root.withdraw()
    processed_chips_folder_path = filedialog.askdirectory(title="Select folder with Chip Excel files")

if not processed_chips_folder_path:
    raise Exception("No folder selected. Exiting.")

# Constants
workbook_names = [f"Chip {n}" for n in [21,22,23,26,27,32,33,35,36,37,39,40,41,43,44,45,46,47,48,52,53,54,55,56,57,59]]
worksheet_names = [
    "0pM_asso", "0pM_disso",
    "100pM_asso", "100pM_disso",
    "1nM_asso", "1nM_disso",
    "10nM_asso", "10nM_disso",
    "100nM_asso", "100nM_disso"
]

CHIP_INFO = {
    'Chip 21': ["Cas_complex", "0.5", "HU"], 'Chip 22': ["Cas_complex", "0.5", "HU"],
    'Chip 23': ["Cas_complex", "5", "SCDU"], 'Chip 26': ["Cas_complex", "5", "HU"],
    'Chip 27': ["Cas_complex", "5", "HU"], 'Chip 32': ["Cas_complex", "5", "HU"],
    'Chip 33': ["Cas_only", "1", "HU"], 'Chip 35': ["Cas_complex", "1", "HU"],
    'Chip 36': ["Cas_only", "1", "HU"], 'Chip 37': ["Cas_complex", "1", "HU"],
    'Chip 39': ["Cas_complex", "1", "HU"], 'Chip 40': ["Cas_complex", "1", "HU"],
    'Chip 41': ["Cas_complex", "1", "SCDU"], 'Chip 43': ["Cas_complex", "5", "SCDU"],
    'Chip 44': ["Cas_complex", "5", "SCDU"], 'Chip 45': ["Cas_only", "5", "HU"],
    'Chip 46': ["Cas_complex", "1", "SCDU"], 'Chip 47': ["Cas_only", "1", "HU"],
    'Chip 48': ["cas_complex", "5", "SCDU"], 'Chip 52': ["Cas_complex", "5", "HU"],
    'Chip 53': ["Cas_complex", "1", "SCDU"], 'Chip 54': ["Cas_only", "5", "HU"],
    'Chip 55': ["Cas_complex", "1", "SCDU"], 'Chip 56': ["Cas_only", "5", "HU"],
    'Chip 57': ["Cas_only", "5", "HU"], 'Chip 59': ["Cas_complex", "0.5", "HU"]
}

def plot_linear_regression(X, y, model, chip_name, sheet, y_label, plot_boolean=False):
    if not plot_boolean:
        return
    plt.figure(figsize=(8, 5))
    plt.scatter(X, y, color="blue", label="Data")
    X_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
    y_line = model.predict(X_line)
    plt.plot(X_line, y_line, color="red", linewidth=2, label="Fit")
    plt.xlabel("Time (mins)")
    plt.ylabel(y_label)
    plt.title(f"{chip_name} - {sheet}\n{y_label} (Slope: {model.coef_[0]:.2f}, R²: {model.score(X, y):.2f})")
    plt.legend()
    plt.grid(True)
    plt.show()

results = []

for file_name in os.listdir(processed_chips_folder_path):
    file_path = os.path.join(processed_chips_folder_path, file_name)
    chip_name = os.path.splitext(file_name)[0]

    if chip_name not in workbook_names or not file_path.endswith(".xlsx"):
        continue

    xl = pd.ExcelFile(file_path)

    df_0_asso = xl.parse("0pM_asso") if "0pM_asso" in xl.sheet_names else None
    df_0_disso = xl.parse("0pM_disso") if "0pM_disso" in xl.sheet_names else None

    for sheet in xl.sheet_names:
        if sheet not in worksheet_names:
            continue

        df = xl.parse(sheet)
        df.columns = df.columns.str.strip()  # clean up column headers

        if df.empty:
            continue

        # Linear regression for slopes
        try:
            X = df['time(mins)'].values.reshape(-1, 1)
            y_a = df['delta Rct-a'].values
            model_a = LinearRegression().fit(X, y_a)
            slope_a = model_a.coef_[0]
            r2_a = model_a.score(X, y_a)
            plot_linear_regression(X, y_a, model_a, chip_name, sheet, 'delta Rct-a', plot_boolean)
        except Exception as e:
            slope_a, r2_a = np.nan, np.nan

        try:
            y_d = df['delta Rct-d'].values
            model_d = LinearRegression().fit(X, y_d)
            slope_d = model_d.coef_[0]
            r2_d = model_d.score(X, y_d)
            plot_linear_regression(X, y_d, model_d, chip_name, sheet, 'delta Rct-d', plot_boolean)
        except Exception as e:
            slope_d, r2_d = np.nan, np.nan

        # --- Subtraction vs 0pM, matched by chip and phase ---
        delta_a_minus_0, delta_d_minus_0 = np.nan, np.nan

        chip_type, mgcl_conc, protein = CHIP_INFO.get(chip_name, ["", "", ""])
        results.append({
            "Chip Name": chip_name,
            "Type": chip_type,
            "Concentration (MgCl)": mgcl_conc,
            "Protein": protein,
            "Concentration": sheet,
            "delta Rct-a Slope": slope_a,
            "R^2_a": r2_a,
            "delta Rct-d Slope": slope_d,
            "R^2_d": r2_d,
            "delta Rct-a concentration minus zero concentration": delta_a_minus_0,
            "delta Rct-d concentration minus zero concentration": delta_d_minus_0
        })

# After building results list
output_df = pd.DataFrame(results)

# --- Compute delta minus 0pM per chip ---
output_df['delta Rct-a concentration minus zero concentration'] = np.nan
output_df['delta Rct-d concentration minus zero concentration'] = np.nan

chips = output_df['Chip Name'].unique()
for chip in chips:
    chip_rows = output_df[output_df['Chip Name'] == chip].index

    # Reference values for this chip
    asso_ref_a = output_df.loc[chip_rows[0], 'delta Rct-a Slope']   # first 0pM_asso
    asso_ref_d = output_df.loc[chip_rows[0], 'delta Rct-d Slope']
    disso_ref_a = output_df.loc[chip_rows[1], 'delta Rct-a Slope']  # first 0pM_disso
    disso_ref_d = output_df.loc[chip_rows[1], 'delta Rct-d Slope']

    for i, idx in enumerate(chip_rows):
        if i % 2 == 0:  # asso
            output_df.loc[idx, 'delta Rct-a concentration minus zero concentration'] = output_df.loc[idx, 'delta Rct-a Slope'] - asso_ref_a
            output_df.loc[idx, 'delta Rct-d concentration minus zero concentration'] = output_df.loc[idx, 'delta Rct-d Slope'] - asso_ref_d
        else:           # disso
            output_df.loc[idx, 'delta Rct-a concentration minus zero concentration'] = output_df.loc[idx, 'delta Rct-a Slope'] - disso_ref_a
            output_df.loc[idx, 'delta Rct-d concentration minus zero concentration'] = output_df.loc[idx, 'delta Rct-d Slope'] - disso_ref_d

# Save updated Excel file
output_path = os.path.join(processed_chips_folder_path, "big data sheet for Deepta.xlsx")
output_df.to_excel(output_path, index=False)
print(f"Saved results with delta minus 0pM to {output_path}")

print("Done.")

Saved results with delta minus 0pM to c:\Users\austi\OneDrive - UC San Diego\Documents\AranLab\BioSensorAnalysisSrcCode\EIS_Analysis\Austin_EIS\MegaExcel\Processed Chips for PCAI1ia\big data sheet for Deepta.xlsx
Done.
